# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [3]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5051 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [5]:
import functools

# TODO: Implementacja dekoratora
def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, list):
                print(f"[Dekorator] znaleziono listę w args, liczba elementów {len(arg)}")
        
        for value in kwargs.values():
            if isinstance(value, list):
                print(f"[Dekorator] znaleziono listę w kwargs, liczba elementów {len(value)}")
        return func(*args, **kwargs)
    return wrapper

# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

print("Test 1")
process_data([10,20,30], "Dane A")

print("Test 2")
process_data(name="Dane B", data_list=["jabłko", "banan"])

Test 1
[Dekorator] znaleziono listę w args, liczba elementów 3
Przetwarzanie Dane A
Test 2
[Dekorator] znaleziono listę w kwargs, liczba elementów 2
Przetwarzanie Dane B


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [10]:
import functools
from datetime import datetime
import time

# TODO: Implementacja dekoratora z argumentem
def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            start_time = time.perf_counter()

            result = func(*args, **kwargs)

            end_time = time.perf_counter()
            execution_time = end_time - start_time

            with open(filename, "a", encoding="utf-8") as file:
                log_line = f"[{current_date}] Funkcja: {func.__name__} | Czas wykonania: {execution_time:.4f}s\n"
                file.write(log_line)
            return result
        return wrapper
    return decorator

@logger("app_behavior.log")
def simulate_heavy_work(seconds):
    print(f"rozpoczynam pracę na {seconds} sekund...")
    time.sleep(seconds)
    return "praca skocnzona"

print(simulate_heavy_work(1.5))

rozpoczynam pracę na 1.5 sekund...
praca skocnzona


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [6]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [12]:
# TODO: Implementacja deskryptora logującego dostęp
class AccessLogger:
    def __set_name__(self, owner, name):
        self.name = name
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        
        print(F"[LOG] odczyt atrybuty '{self.name}'")

        return instance.__dict__.get(self.name)
    
    def __set__(self, instance, value):
        print(f"[LOG] Zapis atrybutu '{self.name}' na wartość: {value}")

        instance.__dict__[self.name] = value

class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

if __name__ == "__main__":
        print("Tworzenie użytkownika (wywołanie __set__ dla imienia i wieku)")
        user = Uzytkownik("Janek", 25)

        print("Odczyt danych (wywołanie __get__)")
        print(f"Imie uzytkownika{user.imie}")

        print("Modyfikacja danych (wywołanie __set__)")
        user.wiek = 31

        print("Ponowny odczyt zmodyfikowanych danych (wywolanie __get__)")
        print(f"Wiek użytkownika {user.wiek}")

Tworzenie użytkownika (wywołanie __set__ dla imienia i wieku)
[LOG] Zapis atrybutu 'imie' na wartość: Janek
[LOG] Zapis atrybutu 'wiek' na wartość: 25
Odczyt danych (wywołanie __get__)
[LOG] odczyt atrybuty 'imie'
Imie uzytkownikaJanek
Modyfikacja danych (wywołanie __set__)
[LOG] Zapis atrybutu 'wiek' na wartość: 31
Ponowny odczyt zmodyfikowanych danych (wywolanie __get__)
[LOG] odczyt atrybuty 'wiek'
Wiek użytkownika 31


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [13]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
# TODO: Implementacja generatora ciągu Collatza
def collatz_generator(n):
    if n < 1:
        return
    
    while n > 1:
        yield n

        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
    yield n

# Test
if __name__ == "__main__":
    print("Test 1: Ciąg collatza dla liczby 10")
    for status in collatz_generator(10):
        print(status)

    print("Test 2: Ciąg collatza dla liczby 7")
    wynik_lista = list(collatz_generator(7))
    print(wynik_lista)


Test 1: Ciąg collatza dla liczby 10
10
5
16
8
4
2
1
Test 2: Ciąg collatza dla liczby 7
[7, 22, 11, 34, 17, 52, 26, 13, 40, 20, 10, 5, 16, 8, 4, 2, 1]


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [15]:
current_user = {"username": "admin", "role": "superuser"}

# TODO: Implementacja dekoratora @require_role
def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            user_role = current_user.get("role")

            if user_role != role:
                raise PermissionError(F"Brak dostępu! Wymagana rola '{role}', twoja rola '{user_role}'")

            return func(*args, **kwargs)
        return wrapper
    return decorator

if __name__ == "__main__":
    @require_role("superuser")
    def delete_database():
        return "Baza danych została pomyślnie usunięta"

    @require_role("moderator")
    def ban_user(username):
        return f"Użytkownik {username} został zbanowany."

    print("Test 1: wywolanie funkcji z poprawna rola")
    try:
        wynik = delete_database()
        print(wynik)
    except PermissionError as e:
        print(f"Przechwycono oczekiwany błąd: {e}")

Test 1: wywolanie funkcji z poprawna rola
Baza danych została pomyślnie usunięta


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
# TODO: Implementacja deskryptora Typed
class Typed:
    pass

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [ ]:
# TODO: Implementacja generatora liczb pierwszych
def prime_generator():
    pass